In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from process_data import process_data, window_normalize
from pathlib import Path

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

In [8]:
from DeepLOB import create_dataloader, DeepLOB, train_model, validate_engine, PnLFocalLoss, EarlyStopping

In [ ]:
# Define Hyperparameter
alpha = 0.5 * 1e-4  #bps
FE = False
label_window = 10   #ticks
window_size = 100  #ticks
normalize_window = 5 #days

In [ ]:
inputpath = 'Data/Raw_data/'
outputpath = 'Data/DeepLOB_data/Processed_data/'
path = Path(outputpath)
path.mkdir(parents=True, exist_ok=True)

raw_files = list(Path(inputpath).glob('*.csv'))

for raw_file in raw_files:
    print(f'Processing file:{raw_file.name}')
    outputfile = f'{outputpath}processed_{raw_file.stem[-4:]}.csv'
    process_data(raw_file, outputfile, label_method='l2', label_window=label_window, FE = FE, alpha = alpha)

inputpath_normalized = 'Data/DeepLOB_data/Processed_data/'
outputpath_normalized = 'Data/DeepLOB_data/Normalized_data/' 
path = Path(outputpath_normalized)
path.mkdir(parents=True, exist_ok=True)
window_normalize(inputpath_normalized, outputpath_normalized, FE = FE, window_size = 5)

Processing file:IM2606_20260401.csv
Data processing completed. Processed data saved to Data/DeepLOB_data/Processed_data/processed_0401.csv.
Processing file:IM2606_20260402.csv
Data processing completed. Processed data saved to Data/DeepLOB_data/Processed_data/processed_0402.csv.
Processing file:IM2606_20260403.csv
Data processing completed. Processed data saved to Data/DeepLOB_data/Processed_data/processed_0403.csv.
Processing file:IM2606_20260407.csv
Data processing completed. Processed data saved to Data/DeepLOB_data/Processed_data/processed_0407.csv.
Processing file:IM2606_20260408.csv
Data processing completed. Processed data saved to Data/DeepLOB_data/Processed_data/processed_0408.csv.
Processing file:IM2606_20260409.csv
Data processing completed. Processed data saved to Data/DeepLOB_data/Processed_data/processed_0409.csv.
Processing file:IM2606_20260410.csv
Data processing completed. Processed data saved to Data/DeepLOB_data/Processed_data/processed_0410.csv.
Processing file:IM26

In [ ]:
inputpath = 'Data/DeepLOB_data/Normalized_data'

train_loader, label_train = create_dataloader(inputpath, start_files=0, num_files=24, window_size=window_size, target_size=1, batch_size=512, shuffle=True)

val_loader, label_val = create_dataloader(inputpath, start_files=24, num_files=4, window_size=window_size, target_size=1, batch_size=32, shuffle=False, drop_last = False)

test_loader, label_test = create_dataloader(inputpath, start_files=28, num_files=3, window_size=window_size, target_size=1, batch_size=32, shuffle=False, drop_last = False)

label_train['Pctg%'] = round(label_train['Count']/label_train['Count'].sum() * 100, 2)
print(label_train)

Found 24 CSV files for this dataloader.

Total days used for training: 24
Total samples in final dataset: 671857

Sample X shape: torch.Size([100, 20]), Sample Y shape: torch.Size([1])

Batch Size: 512
Total number of batches per epoch: 1312

Data shape: torch.Size([512, 100, 20])
Label shape: torch.Size([512, 1])

Label distribution across all training files:
          Count
Down     194940
Neutral  280737
Up       198556
Found 4 CSV files for this dataloader.

Total days used for training: 4
Total samples in final dataset: 114079

Sample X shape: torch.Size([100, 20]), Sample Y shape: torch.Size([1])

Batch Size: 64
Total number of batches per epoch: 1782

Data shape: torch.Size([64, 100, 20])
Label shape: torch.Size([64, 1])

Label distribution across all training files:
         Count
Down     40537
Neutral  33026
Up       40912
Found 3 CSV files for this dataloader.

Total days used for training: 3
Total samples in final dataset: 85311

Sample X shape: torch.Size([200, 20]), Sampl

In [ ]:
# Define training parameter
num_features = 5
num_epochs = 200

learning_rate = 0.0005

total_steps = len(train_loader) * num_epochs
warmup_steps = int(total_steps * 0.05)

class_weights = label_train['Count'].sum() / (label_train['Count'] * len(label_train))
weights = torch.tensor(class_weights.values, dtype=torch.float32)
print(f"\nClass weights for CrossEntropyLoss:{weights}")

pnl_matrix = [
    [0.90, 0.10, 0.00],
    [0.15, 0.70, 0.15],
    [0.00, 0.10, 0.90]
]

In [ ]:
loss_history = []

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DeepLOB_model = DeepLOB(num_features=num_features, num_classes=3)
DeepLOB_model.to(device)

optimizer = torch.optim.AdamW(DeepLOB_model.parameters(), lr=learning_rate, weight_decay=1e-3)

lr_scheduler = SequentialLR(
    optimizer,
    schedulers=[
        LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps),
        CosineAnnealingLR(optimizer, T_max = total_steps - warmup_steps, eta_min=5e-5)
    ],
    milestones = [warmup_steps]
)

criterion = PnLFocalLoss(soft_targets=pnl_matrix, gamma=2.0, reduction='mean', device = device)

early_stopping = EarlyStopping(patience = 20, verbose = True, path = 'Model/cache_/deeplob_best_weights.pth', monitor_loss = True)

In [ ]:
print('Start training...')

for epoch in range(num_epochs):
    # Training
    avg_loss, avg_acc = train_model(DeepLOB_model, train_loader, optimizer, criterion, device, lr_scheduler=lr_scheduler)
    loss_history.append(avg_loss)
    print(f'Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}, Average Accuracy: {avg_acc:.4f}')

    # Validation
    avg_val_loss, avg_val_accuracy, _, _ = validate_engine(DeepLOB_model, val_loader, criterion, device)
    early_stopping(avg_val_loss, DeepLOB_model)

    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

print('Training Ends')

DeepLOB_model.load_state_dict(torch.load('Model/cache_/deeplob_best_weights.pth', weights_only=True))
print("Best model weights loaded.")
DeepLOB_model.eval()